In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-3.5-large"
PROMPTS = ["A professional photo of a cat"]
NEGATIVE_PROMPT = ""
INDEX_OFFSET = 0
IMG_SIZE = 1024
STEPS = 40
GUIDANCE = 4.5
PRECISION = "fp16"
OUTPUT_DIR = "images"

import traceback

try:
    # 1. Imports
    import json
    import os
    import gc
    import importlib
    import inspect
    import subprocess
    import sys
    import torch
    from tqdm.auto import tqdm

    # 2. Environment setup
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 3. Load HF_TOKEN
    HF_TOKEN = None
    TOKEN_PATH = "/kaggle/input/imggenhub-hf-token/hf_token.json"
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, "r") as f:
            HF_TOKEN = json.load(f).get("HF_TOKEN")

    # 4. Token preview
    if HF_TOKEN:
        print(f"TOKEN PREVIEW: {HF_TOKEN[:5]}...")
    else:
        print("TOKEN MISSING")

    # 5. Detection of model family
    model_family = "unknown"
    model_id_lower = MODEL_ID.lower()
    if "stable-diffusion-3.5" in model_id_lower:
        model_family = "sd35"
    elif "wan2.1" in model_id_lower or "wan-2.1" in model_id_lower:
        model_family = "wan21_chroma"
    elif "qwen" in model_id_lower:
        model_family = "qwen_image"

    print(f"Detected model family: {model_family}")

    # 6. Dependency update logic
    if model_family in ["sd35", "qwen_image", "wan21_chroma"]:
        print("Updating dependencies...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-U", "diffusers==0.36.0", "transformers==4.47.1"], check=True)
        importlib.invalidate_caches()

    import diffusers
    from diffusers import StableDiffusion3Pipeline, AutoPipelineForText2Image

    # 7. Pipeline loading
    print(f"Loading pipeline for {MODEL_ID}...")
    dtype = torch.float16 if PRECISION == "fp16" else torch.bfloat16

    if model_family == "sd35":
        pipe = StableDiffusion3Pipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, token=HF_TOKEN)
    else:
        pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=dtype, token=HF_TOKEN)

    pipe.to("cuda")

    # 8. Generation loop
    print(f"Starting generation for {len(PROMPTS)} prompts...")
    for i, prompt in enumerate(tqdm(PROMPTS)):
        idx = i + INDEX_OFFSET
        filename = os.path.join(OUTPUT_DIR, f"{idx:04d}.png")
        
        image = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            width=IMG_SIZE,
            height=IMG_SIZE,
        ).images[0]
        
        image.save(filename)
        print(f"Saved: {filename}")
        
        # Cleanup
        gc.collect()
        torch.cuda.empty_cache()

except Exception:
    # 9. Error handling
    traceback.print_exc()